In [1]:
!git clone https://github.com/890tsugua/AppliedML2026.git

Cloning into 'AppliedML2026'...
remote: Enumerating objects: 323, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 323 (delta 50), reused 50 (delta 23), pack-reused 229 (from 1)
Receiving objects: 100% (323/323), 79.45 MiB | 28.41 MiB/s, done.
Resolving deltas: 100% (173/173), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd AppliedML2026
!git pull

/content/AppliedML2026
Already up to date.


In [6]:
import os
import zipfile
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

%cd country_cnn
from src.dataset import make_dataloaders_from_dir, make_test_dataloader_from_dir
from src.model import make_model
from src.train import train
from src.evaluate import load_model, evaluate_model


[Errno 2] No such file or directory: 'country_cnn'
/content/AppliedML2026/country_cnn


In [8]:
import zipfile
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/AppliedMLdata2026/AppML_augmented_data.zip"

# Extract into local Colab storage
EXTRACT_DIR = "/content/AppML_augmented_data"

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall("/content")

print("Extraction complete.")

Extraction complete.


In [10]:
# Paths
TRAIN_DIR = "/content/AppML_augmented_data/train_data"
TEST_DIR = "/content/AppML_augmented_data/test_data"

OUTPUT_DIR = Path("/content/drive/MyDrive/AppliedMLdata2026/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Training data:", TRAIN_DIR)
print("Test data:", TEST_DIR)


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# Data loaders
train_loader, val_loader = make_dataloaders_from_dir(
    TRAIN_DIR,
    batch_size=128,
    image_size=224,
    val_split=0.2,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True
)

test_loader = make_test_dataloader_from_dir(
    TEST_DIR,
    batch_size=128,
    image_size=224
)

class_names = train_loader.dataset.dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)


# Train ResNets
model_names = ["resnet50", "resnet101", "resnet152"]

num_epochs = 100
trainable_layers = 1
pretrained = True

summary_results = []

for model_name in model_names:
    print("\n" + "=" * 80)
    print(f"Training {model_name}")
    print("=" * 80)

    save_name = f"{model_name}_model"

    model = make_model(
        model_name=model_name,
        num_classes=num_classes,
        pretrained=pretrained,
        trainable_layers=trainable_layers
    ).to(device)

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        save_name=save_name,
        save_checkpoints=True,
        num_epochs=num_epochs
    )

    history_path = OUTPUT_DIR / f"{model_name}_classification_history.csv"
    pd.DataFrame(history).to_csv(history_path, index=False)

    checkpoint_path = Path("/content/AppliedML2026/country_cnn/country_cnn/outputs/models") / f"{save_name}.pt"

    # Load best saved model
    best_model = make_model(
        model_name=model_name,
        num_classes=num_classes,
        pretrained=False,
        trainable_layers=trainable_layers
    )

    best_model = load_model(best_model, checkpoint_path, device)

    print(f"\nFinal TEST evaluation for {model_name}")

    results = evaluate_model(
        model=best_model,
        dataloader=test_loader,
        device=device,
        class_names=class_names
    )

    top1 = results["top1"]
    top5 = results["top5"]
    preds = results["preds"]
    labels = results["labels"]
    cm = results["confusion_matrix"]
    report = results["classification_report"]

    report_path = OUTPUT_DIR / f"{model_name}_test_classification_report.csv"
    pd.DataFrame(report).transpose().to_csv(report_path)

    cm_path = OUTPUT_DIR / f"{model_name}_test_confusion_matrix.csv"
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)

    pred_path = OUTPUT_DIR / f"{model_name}_test_predictions.csv"
    pd.DataFrame({
        "true_label": labels.numpy(),
        "pred_label": preds.numpy(),
        "true_class": [class_names[i] for i in labels.numpy()],
        "pred_class": [class_names[i] for i in preds.numpy()]
    }).to_csv(pred_path, index=False)

    summary_results.append({
        "model": model_name,
        "test_top1": top1,
        "test_top5": top5,
        "checkpoint": str(checkpoint_path),
        "history": str(history_path),
        "test_classification_report": str(report_path),
        "test_confusion_matrix": str(cm_path),
        "test_predictions": str(pred_path)
    })

    del model
    del best_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


summary_df = pd.DataFrame(summary_results)

summary_path = OUTPUT_DIR / "resnet50_101_152_test_summary.csv"
summary_df.to_csv(summary_path, index=False)

summary_df

Training data: /content/AppML_augmented_data/train_data
Test data: /content/AppML_augmented_data/test_data
Device: cuda
Number of classes: 85

Training resnet50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 182MB/s]
/content/AppliedML2026/country_cnn/src/train.py:88: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
Description:   0%|          | 0/383 [00:00<?, ?it/s]/content/AppliedML2026/country_cnn/src/train.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): # For mixed precision training
Description: 100%|██████████| 96/96 [00:02<00:00, 43.50it/s]


Saved best model
Epoch 1/100 | Train loss: 2.9036, acc: 0.2720, top5: 0.5495 | Val loss: 2.4077, acc: 0.3638, top5: 0.6682


Description: 100%|██████████| 96/96 [00:02<00:00, 46.22it/s]


Saved best model
Epoch 2/100 | Train loss: 2.1337, acc: 0.4231, top5: 0.7331 | Val loss: 2.2399, acc: 0.4017, top5: 0.7100


...

Description: 100%|██████████| 96/96 [00:02<00:00, 45.47it/s]


Epoch 99/100 | Train loss: 0.0127, acc: 0.9974, top5: 1.0000 | Val loss: 2.1122, acc: 0.6017, top5: 0.8550


Description: 100%|██████████| 96/96 [00:02<00:00, 45.31it/s]


Epoch 100/100 | Train loss: 0.0123, acc: 0.9974, top5: 1.0000 | Val loss: 2.0934, acc: 0.6052, top5: 0.8561


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/models/resnet50_model.pt'

In [12]:

# Paths
TEST_DIR = "/content/AppML_augmented_data/test_data"

CHECKPOINT_PATH = "/content/AppliedML2026/country_cnn/outputs/models/resnet50_model.pt"

OUTPUT_DIR = Path("/content/AppliedML2026/country_cnn/outputs/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# Test loader
test_loader = make_test_dataloader_from_dir(
    TEST_DIR,
    batch_size=128,
    image_size=224
)

class_names = test_loader.dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)


# Load ResNet50 best checkpoint
model = make_model(
    model_name="resnet50",
    num_classes=num_classes,
    pretrained=False,
    trainable_layers=1
)

model = load_model(model, CHECKPOINT_PATH, device)


# Evaluate on test data
results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=class_names
)

top1 = results["top1"]
top5 = results["top5"]
preds = results["preds"]
labels = results["labels"]
cm = results["confusion_matrix"]
report = results["classification_report"]


# Save classification report
report_path = OUTPUT_DIR / "resnet50_test_classification_report.csv"
pd.DataFrame(report).transpose().to_csv(report_path)

# Save confusion matrix
cm_path = OUTPUT_DIR / "resnet50_test_confusion_matrix.csv"
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)

# Save predictions
pred_path = OUTPUT_DIR / "resnet50_test_predictions.csv"
pd.DataFrame({
    "true_label": labels.numpy(),
    "pred_label": preds.numpy(),
    "true_class": [class_names[i] for i in labels.numpy()],
    "pred_class": [class_names[i] for i in preds.numpy()]
}).to_csv(pred_path, index=False)

# Save summary
summary_path = OUTPUT_DIR / "resnet50_test_summary.csv"
summary_df = pd.DataFrame([{
    "model": "resnet50",
    "test_top1": top1,
    "test_top5": top5,
    "checkpoint": CHECKPOINT_PATH,
    "classification_report": str(report_path),
    "confusion_matrix": str(cm_path),
    "predictions": str(pred_path)
}])
summary_df.to_csv(summary_path, index=False)

summary_df

Device: cuda
Number of classes: 85
Top-1 accuracy: 0.4869
Top-5 accuracy: 0.7790

Classification report:
                    precision    recall  f1-score   support

           Albania       0.48      0.55      0.51       210
         Argentina       0.26      0.38      0.31       177
         Australia       0.54      0.39      0.45       180
           Austria       0.29      0.26      0.27       180
        Bangladesh       0.67      0.77      0.72       177
           Belgium       0.37      0.13      0.19       180
            Bhutan       0.85      0.67      0.75       180
           Bolivia       0.74      0.36      0.49       180
          Botswana       0.75      0.72      0.74       180
            Brazil       0.34      0.30      0.32       180
          Bulgaria       0.25      0.52      0.34       180
          Cambodia       0.72      0.35      0.47       180
            Canada       0.41      0.33      0.36       180
             Chile       0.31      0.63      0.42     

,model,test_top1,test_top5,checkpoint,classification_report,confusion_matrix,predictions
0,resnet50,0.486883,0.778974,/content/AppliedML2026/country_cnn/outputs/mod...,/content/AppliedML2026/country_cnn/outputs/mod...,/content/AppliedML2026/country_cnn/outputs/mod...,/content/AppliedML2026/country_cnn/outputs/mod...


In [15]:
import shutil

src = "/content/drive/MyDrive/Colab Notebooks/Training_loop.ipynb"
dst = "/content/drive/MyDrive/AppliedML2026/country_cnn/Training_loop.ipynb"

shutil.copy2(src, dst)

print("Notebook copied.")

cp: cannot create regular file '/content/drive/MyDrive/AppliedML2026/country_cnn/Notebooks': No such file or directory
